In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pygris import tracts 
from pygris.data import get_lodes
import matplotlib.pyplot as plt


In [2]:
NY_2014_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2014.csv")
NY_2015_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2015.csv")
NY_2016_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2016.csv")

In [4]:
hudson_2014=(
    NY_2014_commute_df
    .assign(w_trct = NY_2014_commute_df["w_geocode"]
            .astype(str)
            .str.slice(stop=11),

            h_trct = NY_2014_commute_df["h_geocode"]
            .astype(str)
            .str.slice(stop=11)
            )
    .query('w_trct=="36061009903"')
    .groupby("h_trct", as_index =False)
    .agg({"S000":sum})
    .rename({'S000': 'hudson_workers'}, axis = 1)
    .assign(w_trct="36061009903")        
)


hudson_2016=(
    NY_2016_commute_df
    .assign(w_trct = NY_2016_commute_df["w_geocode"]
            .astype(str)
            .str.slice(stop=11),

            h_trct = NY_2016_commute_df["h_geocode"]
            .astype(str)
            .str.slice(stop=11)
            )
    .query('w_trct=="36061009903"')
    .groupby("h_trct", as_index =False)
    .agg({"S000":sum})
    .rename({'S000': 'hudson_workers'}, axis = 1)
    .assign(w_trct="36061009903")        
)


C:\Users\fiodo\AppData\Local\Temp\ipykernel_19916\4079403139.py:13: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({"S000":sum})
C:\Users\fiodo\AppData\Local\Temp\ipykernel_19916\4079403139.py:31: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({"S000":sum})


In [5]:
hudson_2014_per_1000=(
    NY_2014_commute_df
    .assign(
        h_trct = NY_2014_commute_df["h_geocode"]
            .astype(str)
            .str.slice(stop=11)
    )
    .groupby("h_trct", as_index=False)
    .agg({"S000":sum})
    .rename({'S000': 'total_workers'}, axis = 1)
    .merge(hudson_2014, on="h_trct")
    .assign(hudson_workers_per_1000=lambda row: 1000*(row["hudson_workers"]/row["total_workers"]))
)


hudson_2016_per_1000=(
    NY_2016_commute_df
    .assign(
        h_trct = NY_2016_commute_df["h_geocode"]
            .astype(str)
            .str.slice(stop=11)
    )
    .groupby("h_trct", as_index=False)
    .agg({"S000":sum})
    .rename({'S000': 'total_workers'}, axis = 1)
    .merge(hudson_2016, on="h_trct")
    .assign(hudson_workers_per_1000=lambda row: 1000*(row["hudson_workers"]/row["total_workers"]))
)



C:\Users\fiodo\AppData\Local\Temp\ipykernel_19916\3686399125.py:9: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({"S000":sum})
C:\Users\fiodo\AppData\Local\Temp\ipykernel_19916\3686399125.py:24: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({"S000":sum})


In [3]:
NY_tracts = tracts(state = "NY", cb = True, 
                   county=['Bronx County', 'Kings County', 'New York County', 'Queens County', 'Richmond County'
],
                    year = 2020, cache = True)
#NY_tracts = NY_tracts[NY_tracts['ALAND'] > 0]
#NY_tracts = NY_tracts[NY_tracts['AWATER'] / (NY_tracts['AWATER'] + NY_tracts['ALAND']) < 0.9]


Using FIPS code '36' for input 'NY'
Using FIPS code '005' for input 'Bronx County'
Using FIPS code '047' for input 'Kings County'
Using FIPS code '061' for input 'New York County'
Using FIPS code '081' for input 'Queens County'
Using FIPS code '085' for input 'Richmond County'


In [6]:
hudson_2014_per_1000_geom = NY_tracts.merge(hudson_2014_per_1000, left_on = "GEOID", right_on = "h_trct",
                             how = "left")

w_geom = NY_tracts[["GEOID", "geometry"]].rename(columns={"GEOID": "w_trct", "geometry": "w_geometry"})


hudson_2014_per_1000_geom.fillna(0, inplace = True)
hudson_2014_per_1000_geom_w=hudson_2014_per_1000_geom.merge(w_geom, on="w_trct",
                             how = "left")
hudson_2014_per_1000_geom_w


hudson_2016_per_1000_geom = NY_tracts.merge(hudson_2016_per_1000, left_on = "GEOID", right_on = "h_trct",
                             how = "left")

w_geom = NY_tracts[["GEOID", "geometry"]].rename(columns={"GEOID": "w_trct", "geometry": "w_geometry"})


hudson_2016_per_1000_geom.fillna(0, inplace = True)
hudson_2016_per_1000_geom_w=hudson_2016_per_1000_geom.merge(w_geom, on="w_trct",
                             how = "left")
hudson_2016_per_1000_geom_w

,STATEFP,COUNTYFP,TRACTCE,AFFGEOID,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,LSAD,ALAND,AWATER,geometry,h_trct,total_workers,hudson_workers,w_trct,hudson_workers_per_1000,w_geometry
0,36,005,009200,1400000US36005009200,36005009200,92,Census Tract 92,NY,Bronx County,New York,CT,310475,0,"POLYGON ((-73.86034 40.83329, -73.85906 40.833...",36005009200,1703.0,1.0,36061009903,0.587199,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
1,36,005,006400,1400000US36005006400,36005006400,64,Census Tract 64,NY,Bronx County,New York,CT,167185,0,"POLYGON ((-73.87395 40.83424, -73.8738 40.8359...",36005006400,1327.0,1.0,36061009903,0.753580,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
2,36,005,036501,1400000US36005036501,36005036501,365.01,Census Tract 365.01,NY,Bronx County,New York,CT,95827,0,"POLYGON ((-73.88832 40.84518, -73.88776 40.844...",36005036501,1344.0,3.0,36061009903,2.232143,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
3,36,005,007900,1400000US36005007900,36005007900,79,Census Tract 79,NY,Bronx County,New York,CT,197933,0,"POLYGON ((-73.90822 40.8162, -73.90759 40.8166...",36005007900,2022.0,1.0,36061009903,0.494560,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
4,36,005,024700,1400000US36005024700,36005024700,247,Census Tract 247,NY,Bronx County,New York,CT,149162,0,"POLYGON ((-73.92096 40.85511, -73.92 40.85668,...",0,0.0,0.0,0,0.000000,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2319,36,047,095800,1400000US36047095800,36047095800,958,Census Tract 958,NY,Kings County,New York,CT,234373,0,"POLYGON ((-73.91519 40.64286, -73.91444 40.643...",36047095800,1518.0,2.0,36061009903,1.317523,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
2320,36,061,008602,1400000US36061008602,36061008602,86.02,Census Tract 86.02,NY,New York County,New York,CT,106507,317001,"POLYGON ((-73.96932 40.74952, -73.96891 40.750...",36061008602,122.0,1.0,36061009903,8.196721,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
2321,36,081,071305,1400000US36081071305,36081071305,713.05,Census Tract 713.05,NY,Queens County,New York,CT,135001,0,"POLYGON ((-73.85351 40.72536, -73.85245 40.726...",36081071305,2285.0,1.0,36061009903,0.437637,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
2322,36,081,141700,1400000US36081141700,36081141700,1417,Census Tract 1417,NY,Queens County,New York,CT,892280,0,"POLYGON ((-73.79605 40.75005, -73.79472 40.750...",0,0.0,0.0,0,0.000000,None


In [7]:
# Make sure both geometries are in a projected CRS (e.g., meters)
gdf_2014 = hudson_2014_per_1000_geom_w.to_crs(epsg=2263)  # NY State Plane, in feet (or use 3857 for meters)
gdf_2014["w_geometry"] = gdf_2014["w_geometry"].to_crs(epsg=2263)

# Calculate distance from h_trct geometry to w_trct geometry
gdf_2014["tract_distance"] = gdf_2014.geometry.distance(gdf_2014["w_geometry"])
gdf_2014["distance_km"] = (gdf_2014["tract_distance"] * 0.0003048 ).round(2)

# Make sure both geometries are in a projected CRS (e.g., meters)
gdf_2016 = hudson_2016_per_1000_geom_w.to_crs(epsg=2263)  # NY State Plane, in feet (or use 3857 for meters)
gdf_2016["w_geometry"] = gdf_2016["w_geometry"].to_crs(epsg=2263)

# Calculate distance from h_trct geometry to w_trct geometry
gdf_2016["tract_distance"] = gdf_2016.geometry.distance(gdf_2016["w_geometry"])
gdf_2016["distance_km"] = (gdf_2016["tract_distance"] * 0.0003048 ).round(2)


In [8]:
#Create 5 km bins and labels
bin_edges = range(0, int(max(gdf_2014["distance_km"].max(), gdf_2016["distance_km"].max())) + 10, 5)
bin_labels = [f"{i}-{i+5} km" for i in bin_edges[:-1]]

# Bin both datasets
gdf_2014["distance_bin_5km"] = pd.cut(gdf_2014["distance_km"], bins=bin_edges, labels=bin_labels)
gdf_2016["distance_bin_5km"] = pd.cut(gdf_2016["distance_km"], bins=bin_edges, labels=bin_labels)
#gdf_2015["distance_bin_5km"] = pd.cut(gdf_2016["distance_km"], bins=bin_edges, labels=bin_labels)


# Function to compute summary with 95% CI
def compute_summary_with_ci(df, label):
    grouped = (
        df.groupby("distance_bin_5km")["hudson_workers_per_1000"]
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .rename(columns={'mean': 'avg', 'std': 'stddev', 'count': 'n'})
    )
    grouped['ci95'] = 1.96 * grouped['stddev'] / np.sqrt(grouped['n'])
    grouped['year'] = label
    return grouped

# Compute summaries
summary_2014 = compute_summary_with_ci(gdf_2014, "2014")
#summary_2015 = compute_summary_with_ci(gdf_2015, "2015")
summary_2016 = compute_summary_with_ci(gdf_2016, "2016")

# Create shaded confidence bands
fig = go.Figure()

for summary_df, color, fill in zip(
    [summary_2014,  summary_2016],
    ["blue", "orange", "green"],
    ["rgba(0, 0, 255, 0.2)", "rgba(255, 165, 0, 0.2)", "rgba(0, 128, 0, 0.2)"]
):
    label = summary_df["year"].iloc[0]

    # Line and markers
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_5km"],
        y=summary_df["avg"],
        mode="lines+markers",
        name=label,
        line=dict(color=color)
    ))

    # Upper bound (invisible line)
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_5km"],
        y=summary_df["avg"] + summary_df["ci95"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Lower bound (fills to upper line)
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_5km"],
        y=summary_df["avg"] - summary_df["ci95"],
        mode="lines",
        line=dict(width=0),
        fill='tonexty',
        fillcolor=fill,
        hoverinfo='skip',
        showlegend=False
    ))


# Layout
fig.update_layout(
    title="Average Hudson Commuters per 5km Distance Bin (with 95% CI)",
    xaxis_title="Distance Bin (km)",
    yaxis_title="Avg. Commuters per 1000",
    template="plotly_white",
    yaxis=dict(range=[0, 3.5])
)

fig.show()

C:\Users\fiodo\AppData\Local\Temp\ipykernel_19916\3968841143.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("distance_bin_5km")["hudson_workers_per_1000"]
C:\Users\fiodo\AppData\Local\Temp\ipykernel_19916\3968841143.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("distance_bin_5km")["hudson_workers_per_1000"]
